# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Setup (local Cursor)

In [5]:
import os
from pathlib import Path

import duckdb

# Find repo root 
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Load HF_TOKEN from repo .env 
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    import getpass
    print("Tip: pip install python-dotenv  (or set HF_TOKEN in your shell)")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected. Paths ready:")
print("  FACT_MAR = month=2026-03 (proof queries + five-feature frame)")

Connected. Paths ready:
  FACT_MAR = month=2026-03 (proof queries + five-feature frame)


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane 4 — CTR / Engagement Opportunity Scoring**

**1. What one row means**  
One row = **one web page** (`content_hash_id`) for one client (`client_hash_id`), after I roll up its daily search stats into page-level numbers.

**2. Which table(s)**  
- `fact_content_daily_performance` — daily impressions, clicks, position, sessions  
- `dim_content` — page metadata (join on `content_hash_id`)  
- `dim_clients` — client history / GA4 start dates (join on `client_hash_id`)

**3. Time window**  
- **ML-04 slice:** **`month=2026-03`** only — proof queries and the five-feature frame (mid-panel; not `_sample`, which is June 2026)  
- **Decision moment:** end of March 2026 — “as of March 31, should a reviewer look at this page first?”  
- **Later (capstone):** extend to a full 90-day window ending at the decision date

**4. What I predict / rank (label / proxy)**  
`is_ctr_underperformer` = 1 when a page’s `ctr_mar` is **below the middle CTR of its `position_tier`**, else 0.  
Only on pages with `imp_mar >= 100` and a real Google position (`pos_avg_mar > 0`; remember `gsc_avg_position = 0` means “no data”).

**5. One thing I deliberately exclude**  
Engagement numbers from rows where **`ga4_data_available IS NOT TRUE`**. Before GA4 existed for a client, zeros mean “no tracking,” not “no engagement.” I filter with `IS TRUE`, not `= FALSE`.

**6. Output (one sentence)**  
A ranked list of pages where a reviewer should look first for CTR fix opportunities — decision-support, not a guarantee.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (max 5 — knowable at the decision moment)**

| Feature | Plain meaning | Knowable at decision moment because… |
|---|---|---|
| `imp_mar` | Google impressions in March 2026 | Sums only dates in `month=2026-03` |
| `ctr_mar` | Clicks ÷ impressions × 100 | Same month; no future data |
| `pos_avg_mar` | Average Google rank in March | Same month; `0` rows dropped (no position) |
| `position_tier` | Rank band (`top_3`, `page_1`, `deep`, …) | Derived from `pos_avg_mar` only |
| `engagement_rate_mar` | Engaged sessions ÷ sessions × 100 | Only rows where `ga4_data_available IS TRUE` |

**Label / proxy**  
- `is_ctr_underperformer` — built from `ctr_mar` vs tier median (ranking target for Precision@K later)

**Context (join / split only — never model features)**  
- `content_hash_id`, `client_hash_id`

**Excluded (with why)**  
- `content_hash_id` / `client_hash_id` as features — IDs, not signals  
- `trend_direction`, `trend_pct` — refresh-lane labels; not my lane; leakage risk  
- GA4 metrics when `ga4_data_available IS NOT TRUE` — zeros are fake “no engagement”  
- `ctr_gap` / `tier_median_ctr` — label-derived; used only in the leakage trap, never as features  
- `fact_content_query_90d` for now — fixed 90-day window overlaps recent months; easy leakage  
- Product flags (`health_score`, `needs_ctr_fix`, …) — not in the data on purpose  
- `fact_content_daily_performance_sample` — June 2026 only; sealed test month

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim in Sections 1–2 gets a number below. All three proof queries **and** the five-feature frame use **`month=2026-03`** (`FACT_MAR`) — same month, as the card asks.

Check that printed counts match what you wrote above.

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# ------------------------------------------------------------------
# Query 1 — daily grain on March partition (report_date × client × content)
# Expect: 0 duplicate keys
# ------------------------------------------------------------------
print("=" * 60)
print("QUERY 1 — daily grain check (March 2026)")
grain_daily = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(grain_daily if len(grain_daily) else "OK — 0 duplicate daily keys")

# ------------------------------------------------------------------
# Query 2 — row count + date span on March partition
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("QUERY 2 — March row count + date span")
span = con.sql(f"""
    SELECT
        COUNT(*) AS daily_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FACT_MAR}
""").df()
print(span.to_string(index=False))

# ------------------------------------------------------------------
# Query 3 — GA4 availability (IS TRUE vs IS NOT TRUE)
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("QUERY 3 — ga4_data_available on March rows")
ga4_survive = con.sql(f"""
    SELECT COUNT(*) AS rows_surviving_is_true
    FROM {FACT_MAR}
    WHERE ga4_data_available IS TRUE
""").df()
print(f"Rows surviving ga4_data_available IS TRUE: {int(ga4_survive.loc[0, 'rows_surviving_is_true']):,}")

# ------------------------------------------------------------------
# Five-feature frame — March 2026 only (same month as proof queries)
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("FEATURE FRAME — March 2026 (month=2026-03)")
features = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar,
            SUM(gsc_clicks) AS clk_mar,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_mar,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_mar,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_mar
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    scored AS (
        SELECT
            *,
            CASE WHEN imp_mar > 0 THEN 100.0 * clk_mar / imp_mar END AS ctr_mar,
            CASE
                WHEN sessions_mar > 0 THEN 100.0 * engaged_mar / sessions_mar
            END AS engagement_rate_mar,
            CASE
                WHEN pos_avg_mar <= 3 THEN 'top_3'
                WHEN pos_avg_mar <= 10 THEN 'page_1'
                WHEN pos_avg_mar <= 20 THEN 'striking'
                WHEN pos_avg_mar <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM daily
        WHERE pos_avg_mar > 0
    ),
    labeled AS (
        SELECT
            s.*,
            MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) AS tier_median_ctr,
            CASE
                WHEN ctr_mar < MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM scored s
    )
    SELECT * FROM labeled
""").df()

HONEST_COLS = [
    'content_hash_id', 'imp_mar', 'ctr_mar', 'pos_avg_mar', 'position_tier',
    'engagement_rate_mar', 'is_ctr_underperformer',
]

print(f"Lane slice rows (one row per page): {len(features):,}")
print(f"Share is_ctr_underperformer=1: {features['is_ctr_underperformer'].mean():.3f}")
print(features[HONEST_COLS].head(10).to_string(index=False))

# Aggregated grain check — expect 0 duplicates
dup = features.groupby(['client_hash_id', 'content_hash_id']).size()
dup = dup[dup > 1]
print("\nAggregated grain duplicates:", len(dup), "(expect 0)")

# ------------------------------------------------------------------
# THE TRAP — add one label-derived column, then drop it
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("LEAKAGE TRAP — honest vs cheating score")

model_df = features.dropna(subset=['ctr_mar', 'pos_avg_mar']).copy()
model_df['engagement_rate_mar'] = model_df['engagement_rate_mar'].fillna(0)

honest_feat_cols = ['imp_mar', 'ctr_mar', 'pos_avg_mar', 'engagement_rate_mar']
X_honest = model_df[honest_feat_cols]
y = model_df['is_ctr_underperformer']

X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)
idx_train = X_train.index
idx_test = X_test.index

tree_honest = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_honest.fit(X_honest.loc[idx_train], y_train)
auc_honest = roc_auc_score(y_test, tree_honest.predict_proba(X_honest.loc[idx_test])[:, 1])

# Purposely add label-derived ctr_gap for the trap demo only
ctr_gap = model_df['tier_median_ctr'] - model_df['ctr_mar']
X_leaky = X_honest.assign(ctr_gap=ctr_gap)

tree_leaky = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_leaky.fit(X_leaky.loc[idx_train], y_train)
auc_leaky = roc_auc_score(y_test, tree_leaky.predict_proba(X_leaky.loc[idx_test])[:, 1])

print(f"Honest ROC AUC (no ctr_gap):  {auc_honest:.3f}")
print(f"Leaky ROC AUC (with ctr_gap): {auc_leaky:.3f}")
print(
    "\nWhy: ctr_gap = tier_median_ctr - ctr_mar — almost the same formula as the label."
    "\nThat is leakage: the feature is the answer in disguise. ctr_gap is dropped; keep the honest number."
)

QUERY 1 — daily grain check (March 2026)
OK — 0 duplicate daily keys

QUERY 2 — March row count + date span
 daily_rows  distinct_pages   min_date   max_date
    9841378          331437 2026-03-01 2026-03-31

QUERY 3 — ga4_data_available on March rows
Rows surviving ga4_data_available IS TRUE: 413,966

FEATURE FRAME — March 2026 (month=2026-03)
Lane slice rows (one row per page): 101,441
Share is_ctr_underperformer=1: 0.377
         content_hash_id  imp_mar  ctr_mar  pos_avg_mar position_tier  engagement_rate_mar  is_ctr_underperformer
content_1a0cb7648dc42bd1   4461.0 0.134499     2.761230         top_3                  NaN                      1
content_2bfa1c2bce65b610    422.0 0.236967     5.987917        page_1                  NaN                      0
content_6da207828cf213c8    110.0 0.000000    18.769063      striking                  NaN                      1
content_5cb7083595d4078e   1020.0 0.000000     8.519631        page_1                  NaN                      1
co

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you**
- Whether fixing a title or meta **caused** more clicks (no experiment; observational only)
- What the real keyword, URL, or client name is (everything is hashed)
- True on-page engagement for clients before GA4 started (GSC-only rows)
- Google’s secret ranking rules or algorithm factors
- Whether a refresh will pay off (decision-support ranking, not a guarantee)

**Weaknesses of this slice (March 2026 only)**
- **Unbalanced panel** — clients started tracking at different times
- **CTR noise** — even with `imp_mar >= 100`, low-volume pages wiggle
- **Sparse engagement** — only 4.2% of March daily rows have GA4 (`IS TRUE`)
- **One month only** — not a full 90-day window yet; capstone will widen the slice
- **Proxy label** — `is_ctr_underperformer` compares CTR inside the same month; a stronger capstone uses past features → future CTR change

**Careful words I will use:** observed, measured, directional, decision-support.

In [7]:
# Optional check — clients whose GA4 had not started by end of March 2026
clients = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        SUM(CASE WHEN ga4_data_start IS NULL OR ga4_data_start > DATE '2026-03-31'
                 THEN 1 ELSE 0 END) AS no_ga4_by_march_end
    FROM {DIM_CLIENTS}
""").df()
print(clients.to_string(index=False))

 total_clients  no_ga4_by_march_end
           104                 63.0


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.